In [ ]:
%reload_ext autoreload
%autoreload 2

In [525]:
import pandas as pd

from src import (
    mappings,
)

from persistence import datasets as dataset_persistence

from src.rna import preprocessing as rna_preprocessing
from src.adt import preprocessing as adt_preprocessing

from src.logs import get_logger

log = get_logger()

LEVEL = "celltype.l2"

In [537]:
import warnings
from pandas.errors import PerformanceWarning
from anndata import ImplicitModificationWarning

warnings.simplefilter("ignore", category=PerformanceWarning)
warnings.simplefilter("ignore", category=ImplicitModificationWarning)


In [528]:
# Load the persistence
dataset = dataset_persistence.load_or_create_subsample(subsample_size=10_000,
                                                       level=LEVEL)
rna_dataset, adt_dataset = dataset["rna"], dataset["adt"]

In [531]:
# QC
rna_preprocessing.calculate_qc_metrics_in_place(rna_dataset)

In [532]:
# Normalize
rna_preprocessing.normalize_in_place(rna_dataset)
adt_preprocessing.normalize_in_place(adt_dataset)

In [533]:
# Feature selection

# HV Genes
rna_preprocessing.annotate_highly_variable_genes(rna_dataset)
hv_genes = rna_preprocessing.get_highly_variable_genes(rna_dataset)

protein_names = adt_dataset.var["protein_name"]
marker_genes_for_proteins = mappings.get_marker_genes_for_proteins(
    protein_names)
expressed_genes = rna_dataset.var["gene_name"]

expressed_marker_genes_for_proteins = (marker_genes_for_proteins
                                       .intersection(expressed_genes))

genes_of_interest = expressed_marker_genes_for_proteins.union(hv_genes)

genes_of_interest_mask = rna_dataset.var["gene_name"].isin(
    sorted(genes_of_interest))
rna_of_interest_dataset = rna_dataset[:, genes_of_interest_mask].copy()

In [534]:
# Filtering
rna_of_interest_dataset = rna_preprocessing.apply_basic_filtering(
    rna_of_interest_dataset)

In [538]:
# Scaling
rna_preprocessing.scale_in_place(rna_of_interest_dataset)

# Dependency measures

## 1. Linear marginal - spearman

In [541]:
rna_preprocessing.rank_transform_in_place(rna_of_interest_dataset)

In [542]:
from measures.linear.marginal import spearman_correlation
from rna.preprocessing import LAYER_NAME_RANK_TRANSFORMED

data_df = pd.DataFrame(
    data=rna_of_interest_dataset.layers[LAYER_NAME_RANK_TRANSFORMED],
    index=rna_of_interest_dataset.obs_names,
    columns=rna_of_interest_dataset.var["gene_name"])

target_df = rna_preprocessing.build_target_matrix(rna_of_interest_dataset,
                                                  LEVEL)
spearman_results_df = spearman_correlation.calculate_scores(
    expression_levels_df=data_df,
    labeling_df=target_df
)

In [660]:
from measures.linear.conditional import ledoit_wolf_partial_correlation

data_df = pd.DataFrame(
    data=rna_of_interest_dataset.X,
    index=rna_of_interest_dataset.obs_names,
    columns=rna_of_interest_dataset.var["gene_name"])


ledoit_wolf_results_df = ledoit_wolf_partial_correlation.calculate_scores(
    expression_levels_df=data_df,
    labeling_df=target_df
)

In [ ]:
from measures.non_linear.marginal import mutual_information

data_df = pd.DataFrame(
    data=rna_of_interest_dataset.X,
    index=rna_of_interest_dataset.obs_names,
    columns=rna_of_interest_dataset.var["gene_name"])


mi_results_df = mutual_information.calculate_scores(data_df, target_df)

In [765]:
spearman_results_df

,ASDC,B intermediate,B memory,B naive,CD4 CTL,CD4 Naive,CD4 Proliferating,CD4 TCM,CD4 TEM,CD8 Naive,...,NK Proliferating,NK_CD56bright,Plasmablast,Platelet,Treg,cDC1,cDC2,dnT,gdT,pDC
gene_name,,,,,,,,,,,,,,,,,,,,,
PERM1,-0.001013,-0.002444,-0.002641,-0.003299,-0.002241,-0.004105,-0.001108,-0.003930,-0.002830,-0.003605,...,-0.001673,-0.001920,-0.001505,-0.002409,-0.002461,-0.001204,-0.002461,-0.001498,-0.002713,-0.001876
HES4,-0.008516,-0.025734,-0.046369,-0.057788,-0.035227,-0.053688,-0.016344,-0.066414,-0.052049,-0.059357,...,-0.001728,-0.027964,-0.018245,-0.038850,-0.030658,-0.001146,-0.005059,-0.025081,-0.037113,-0.000221
ISG15,0.003456,-0.018345,-0.042255,-0.078775,-0.004335,-0.076764,0.067700,-0.005583,-0.023281,-0.051989,...,0.026738,-0.013432,0.034409,-0.089250,-0.036390,0.018845,0.045868,0.003682,-0.004658,-0.016602
TNFRSF18,-0.014912,-0.015319,-0.035558,-0.032590,0.001268,-0.075216,0.010485,0.076677,0.038783,-0.069495,...,0.062792,0.192655,-0.001975,-0.040320,-0.018284,-0.023732,-0.046374,-0.026063,-0.011613,-0.028577
TNFRSF4,-0.002467,-0.045077,-0.044237,-0.049693,-0.022641,-0.054521,0.028905,0.234446,0.125649,-0.059714,...,0.010349,0.050414,-0.005897,-0.041828,0.011432,-0.013122,-0.038574,-0.027635,-0.023834,-0.004861
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ITGB2,0.026076,-0.126986,-0.157370,-0.171664,0.034157,-0.184109,0.000667,-0.126035,-0.080853,-0.169176,...,0.098718,0.048640,-0.084461,-0.127287,-0.112656,0.037508,0.143120,-0.030307,0.029438,-0.080531
COL6A2,-0.011634,-0.037220,-0.040977,-0.054953,0.017459,-0.029391,-0.008582,-0.030166,-0.010752,-0.016130,...,0.055138,-0.006862,-0.025954,-0.033792,-0.027132,-0.020767,-0.037585,-0.017771,0.085380,-0.019750
C21orf58,-0.003476,-0.011010,-0.016314,-0.008188,-0.004984,-0.009184,0.210879,-0.020734,-0.020535,-0.006623,...,0.232362,-0.002337,0.032004,0.040165,-0.025155,-0.013914,-0.020556,-0.010237,-0.010398,-0.005422


## Result evaluation / intuition

Before we dive into the formal driver-recovery analysis (AUC_rel) let us look at the results
and try to develop an intuition about its quality.

In [477]:
from src import ground_truth

driver_gene_ground_truth = ground_truth.build_ground_truth(
    adt_dataset, LEVEL, genes_of_interest)

### Heatmap validation

In [630]:
from measures.validation import heatmap as heatmap_validation

spearman_heatmap_results_df = heatmap_validation.calculate_validation_data(
    spearman_results_df)
heatmap_validation.plot(spearman_heatmap_results_df, rna_of_interest_dataset,
                        LEVEL)

In [ ]:
leodit_wolf_heatmap_results_df = heatmap_validation.calculate_validation_data(
    ledoit_wolf_results_df)
heatmap_validation.plot(leodit_wolf_heatmap_results_df, rna_of_interest_dataset,
                        LEVEL)

### Hit/miss validation

In [ ]:
from measures.validation import hit_miss_rates as hit_miss_validation

spearman_hit_miss_validation_df = hit_miss_validation.calculate_validation_data(
    spearman_results_df, driver_gene_ground_truth)

lf_hit_miss_validation_df = hit_miss_validation.calculate_validation_data(lw_df,
                                                                          driver_gene_ground_truth)

In [ ]:
names = ["Spearman", "Ledoit-Wolf", "MI", "MLP"]
summaries = [spearman_hit_miss_validation_df, lf_hit_miss_validation_df,
             lf_hit_miss_validation_df, lf_hit_miss_validation_df]

hit_miss_validation.plot(summaries, names)


In [ ]:
from measures.validation import roc_curve

results = [spearman_results_df, spearman_results_df]
names = ["Spearman", "any"]

roc_curve.plot(results, names, driver_gene_ground_truth)

## Formal method comparison

rank-based driver-recovery metric of CellRank 2 (AUC_rel)


In [ ]:
from measures.benchmarking import driver_recovery_metric

results = [spearman_results_df, spearman_results_df]
names = ["Spearman", "any"]

driver_recovery_metric.compute_auc_rel(spearman_results_df, driver_gene_ground_truth)


driver_recovery_metric.plot(results, names, driver_gene_ground_truth)